In [1]:
%jsroot on

In [ ]:
%jsroot off

In [4]:
// 建议先加载 ROOT 图形
auto c = new TCanvas("c", "TF1 shapes", 800, 600);

// 定义 x 范围（可按需要调整）
double xmin = -7.0;
double xmax =  3.0;

// --- Gaussian-like function ---
TF1 *f_gaus = new TF1(
    "f_gaus",
    "[0]*exp(-0.5*((x-[1])/[2])^2)",
    xmin, xmax
);
f_gaus->SetParameters(0.32, -1.65, 0.75);
f_gaus->SetLineColor(kRed);
f_gaus->SetLineWidth(2);
f_gaus->SetTitle(";x;f(x)");

// --- Quadratic polynomial ---
TF1 *f_poly = new TF1(
    "f_poly",
    "[0]*x*x + [1]*x + [2]",
    xmin, xmax
);
f_poly->SetParameters(-0.01, -0.05, 0.05);
f_poly->SetLineColor(kBlue);
f_poly->SetLineWidth(2);

TF1 *f_sum = new TF1(
    "f_sum",
    "[0]*exp(-0.5*((x-[1])/[2])^2) + [3]*x*x + [4]*x + [5]",
    xmin, xmax
);

f_sum->SetParameters(0.32, -1.65, 0.75,-0.01, -0.05, 0.05);
f_sum->SetLineColor(kMagenta);
f_sum->SetLineWidth(2);

// 先画高斯，再叠加多项式
f_sum->Draw("same");
f_gaus->Draw("same");
f_poly->Draw("same");


// 图例
auto leg = new TLegend(0.60, 0.70, 0.88, 0.88);
leg->AddEntry(f_gaus, "a exp[-(x-#mu)^{2}/2#sigma^{2}]", "l");
leg->AddEntry(f_poly, "c_{1}x^{2}+c_{2}x+c_{3}", "l");
leg->AddEntry(f_sum, "Sum", "l");
leg->Draw();

c->Draw();


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


# File input

In [5]:
    /* ---------- Canvas ---------- */
    auto* c1 = new TCanvas("c1", "Rapidity loss comparison", 800, 600);
    c1->SetTicks();
    c1->SetLeftMargin(0.13);
    c1->SetBottomMargin(0.12);

    /* ---------- Frame ---------- */
    TH1F* frame = c1->DrawFrame(
        -7.5, 0.0,
         0.5, 0.5   // 根据你 h_dNdDyA 实际范围可再调
    );

    frame->SetTitle("");
    frame->GetXaxis()->SetTitle("#Deltay=y-y_{beam}");
    frame->GetYaxis()->SetTitle("(N_{part}/2)dN^{#it{B-#bar{B}}}/d(y-y_{beam})");

    frame->GetXaxis()->SetTitleSize(0.045);
    frame->GetYaxis()->SetTitleSize(0.045);
    frame->GetXaxis()->SetLabelSize(0.040);
    frame->GetYaxis()->SetLabelSize(0.040);

    /* ---------- Legend ---------- */
    auto* leg = new TLegend(0.15, 0.35, 0.48, 0.88);
    leg->SetBorderSize(0);
    leg->SetFillStyle(0);
    leg->SetTextFont(42);
    leg->SetTextSize(0.035);

TFile* fexp = TFile::Open("../expData.root", "READ");
if (!fexp || fexp->IsZombie()) {
    std::cerr << "Cannot open expData.root" << std::endl;
} else {

    const int Nexp = 5;
    TString expNames[Nexp] = {
        "NA49_17p3GeV",
        "BRAHMS_62p4GeV",
        "BRAHMS_200GeV",
        "STAR_62p4GeV",
        "STAR_200GeV"
    };

    TString expLabels[Nexp] = {
        "NA49 Pb+Pb 17.3 GeV",
        "BRAHMS Au+Au 62.4 GeV",
        "BRAHMS Au+Au 200 GeV",
        "STAR Au+Au 62.4 GeV",
        "STAR Au+Au 200 GeV"
    };

    for (int i = 0; i < Nexp; ++i) {

        TGraphErrors* gr =
            (TGraphErrors*)fexp->Get(expNames[i]);

        if (!gr) {
            std::cerr << "Cannot find " << expNames[i]
                      << " in expData.root" << std::endl;
            continue;
        }


        // 直接画，不改任何风格
        gr->Draw("P SAME");

        leg->AddEntry(gr, expLabels[i], "p");
    }
    leg->AddEntry(f_sum, "Fit function", "l");
    leg->Draw();
    f_sum->Draw("same");
    c1->Draw();
}

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


# dN/dy comparison (projectile only)

In [ ]:
auto h_AA_MB_dNdy = (TH1D*) f_AA_MB->Get("h_dNdy");
if(!h_AA_MB_dNdy) { printf("Histogram AAMB not found!\n"); return; }

auto h_AA_05_dNdy = (TH1D*) f_AA_05->Get("h_dNdy");
if(!h_AA_05_dNdy) { printf("Histogram AA05 not found!\n"); return; }

## Normalization+Style

In [ ]:
h_AA_MB_dNdy->Scale(1.0/h_AA_MB_dNdy->Integral());
h_AA_05_dNdy->Scale(1.0/h_AA_05_dNdy->Integral());

h_AA_05_dNdy->SetLineColor(kRed);

h_AA_MB_dNdy->SetLineWidth(2);
h_AA_05_dNdy->SetLineWidth(2);

## Draw

In [ ]:
h_AA_MB_dNdy->GetXaxis()->SetRangeUser(-10,10);

auto c_dNdy = new TCanvas("c1", "dNdy of AA", 800, 800);
h_AA_MB_dNdy->Draw("HIST");
h_AA_05_dNdy->Draw("HIST SAME");

auto legend_dNdy = new TLegend(0.4,0.15,0.6,0.25);
legend_dNdy->AddEntry(h_AA_MB_dNdy, "AA inclusive(MB)", "l");
legend_dNdy->AddEntry(h_AA_05_dNdy, "AA 0~5 % centrality", "l");
legend_dNdy->Draw();
c_dNdy->Draw();

# Draw Ncoll vs dNdy

In [ ]:
auto h2_AA_MB_dNdy_Ncoll = (TH2D*) f_AA_MB->Get("h2_dNdy_Ncoll");
if(!h2_AA_MB_dNdy_Ncoll) { printf("Histogram not found!\n"); return; }

auto h2_AA_05_dNdy_Ncoll = (TH2D*) f_AA_05->Get("h2_dNdy_Ncoll");
if(!h2_AA_05_dNdy_Ncoll) { printf("Histogram not found!\n"); return; }

## For MB

In [ ]:
// Ncoll = 1 
TH1D *h_AA_MB_dNdy_Ncoll1 = h2_AA_MB_dNdy_Ncoll->ProjectionY("h_AA_MB_dNdy_Ncoll1", h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(1), h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(1));

// Ncoll = 2
TH1D *h_AA_MB_dNdy_Ncoll2 = h2_AA_MB_dNdy_Ncoll->ProjectionY("h_AA_MB_dNdy_Ncoll2", h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(2), h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(2));

// Ncoll = 3
TH1D *h_AA_MB_dNdy_Ncoll3 = h2_AA_MB_dNdy_Ncoll->ProjectionY("h_AA_MB_dNdy_Ncoll3", h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(3), h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(3));

// Ncoll > 3
//TH1D *h_AA_MB_dNdy_NcollL = h2_AA_MB_dNdy_Ncoll->ProjectionY("h_AA_MB_dNdy_NcollL", h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(4), h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(25));
TH1D *h_AA_MB_dNdy_NcollL = h2_AA_MB_dNdy_Ncoll->ProjectionY("h_AA_MB_dNdy_NcollL", h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(10), h2_AA_MB_dNdy_Ncoll->GetXaxis()->FindBin(10));

In [ ]:
h_AA_MB_dNdy_Ncoll1->Scale(1.0/h_AA_MB_dNdy_Ncoll1->Integral());
h_AA_MB_dNdy_Ncoll2->Scale(1.0/h_AA_MB_dNdy_Ncoll2->Integral());
h_AA_MB_dNdy_Ncoll3->Scale(1.0/h_AA_MB_dNdy_Ncoll3->Integral());
h_AA_MB_dNdy_NcollL->Scale(1.0/h_AA_MB_dNdy_NcollL->Integral());

h_AA_MB_dNdy_Ncoll1->SetLineColor(kRed);
h_AA_MB_dNdy_Ncoll2->SetLineColor(kBlue);
h_AA_MB_dNdy_Ncoll3->SetLineColor(kGreen);
h_AA_MB_dNdy_NcollL->SetLineColor(kBlack);

h_AA_MB_dNdy_Ncoll1->SetLineWidth(2);
h_AA_MB_dNdy_Ncoll2->SetLineWidth(2);
h_AA_MB_dNdy_Ncoll3->SetLineWidth(2);
h_AA_MB_dNdy_NcollL->SetLineWidth(2);

h_AA_MB_dNdy_Ncoll1->GetXaxis()->SetTitle("y");
h_AA_MB_dNdy_Ncoll2->GetXaxis()->SetTitle("y");
h_AA_MB_dNdy_Ncoll3->GetXaxis()->SetTitle("y");
h_AA_MB_dNdy_NcollL->GetXaxis()->SetTitle("y");

### Draw total

In [ ]:
auto c_dNdy_Ncoll_MB = new TCanvas("c_dNdy_Ncoll_MB", "pA projectile dNdy", 800, 600);

h_AA_MB_dNdy_Ncoll1->GetXaxis()->SetRangeUser(-10,10);

h_AA_MB_dNdy_Ncoll1->Draw("HIST same");
h_AA_MB_dNdy_Ncoll2->Draw("HIST same");
h_AA_MB_dNdy_Ncoll3->Draw("HIST same");
h_AA_MB_dNdy_NcollL->Draw("HIST same");


auto legend_dNdy_Ncoll_MB = new TLegend(0.40,0.7,0.65,0.85);
legend_dNdy_Ncoll_MB->AddEntry(h_AA_MB_dNdy_Ncoll1, "Ncoll = 1", "l");
legend_dNdy_Ncoll_MB->AddEntry(h_AA_MB_dNdy_Ncoll2, "Ncoll = 2", "l");
legend_dNdy_Ncoll_MB->AddEntry(h_AA_MB_dNdy_Ncoll3, "Ncoll = 3", "l");
legend_dNdy_Ncoll_MB->AddEntry(h_AA_MB_dNdy_NcollL, "Ncoll = 10", "l");
legend_dNdy_Ncoll_MB->Draw();
c_dNdy_Ncoll_MB->Draw();

## For b = 0~3.31 fm (correspond to 0~5% centrality)

In [ ]:
// Ncoll = 1 
TH1D *h_AA_05_dNdy_Ncoll1 = h2_AA_05_dNdy_Ncoll->ProjectionY("h_AA_05_dNdy_Ncoll1", h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(1), h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(1));

// Ncoll = 2
TH1D *h_AA_05_dNdy_Ncoll2 = h2_AA_05_dNdy_Ncoll->ProjectionY("h_AA_05_dNdy_Ncoll2", h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(2), h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(2));

// Ncoll = 3
TH1D *h_AA_05_dNdy_Ncoll3 = h2_AA_05_dNdy_Ncoll->ProjectionY("h_AA_05_dNdy_Ncoll3", h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(3), h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(3));

// Ncoll > 3
// TH1D *h_AA_05_dNdy_NcollL = h2_AA_05_dNdy_Ncoll->ProjectionY("h_AA_05_dNdy_NcollL", h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(4), h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(25));
TH1D *h_AA_05_dNdy_NcollL = h2_AA_05_dNdy_Ncoll->ProjectionY("h_AA_05_dNdy_NcollL", h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(10), h2_AA_05_dNdy_Ncoll->GetXaxis()->FindBin(10));

In [ ]:
h_AA_05_dNdy_Ncoll1->Scale(1.0/h_AA_05_dNdy_Ncoll1->Integral());
h_AA_05_dNdy_Ncoll2->Scale(1.0/h_AA_05_dNdy_Ncoll2->Integral());
h_AA_05_dNdy_Ncoll3->Scale(1.0/h_AA_05_dNdy_Ncoll3->Integral());
h_AA_05_dNdy_NcollL->Scale(1.0/h_AA_05_dNdy_NcollL->Integral());

h_AA_05_dNdy_Ncoll1->SetLineColor(kRed);
h_AA_05_dNdy_Ncoll2->SetLineColor(kBlue);
h_AA_05_dNdy_Ncoll3->SetLineColor(kGreen);
h_AA_05_dNdy_NcollL->SetLineColor(kBlack);

h_AA_05_dNdy_Ncoll1->SetLineStyle(9);
h_AA_05_dNdy_Ncoll2->SetLineStyle(9);
h_AA_05_dNdy_Ncoll3->SetLineStyle(9);
h_AA_05_dNdy_NcollL->SetLineStyle(9);

h_AA_05_dNdy_Ncoll1->GetXaxis()->SetTitle("y");
h_AA_05_dNdy_Ncoll2->GetXaxis()->SetTitle("y");
h_AA_05_dNdy_Ncoll3->GetXaxis()->SetTitle("y");
h_AA_05_dNdy_NcollL->GetXaxis()->SetTitle("y");

### Draw total

In [ ]:
auto c_dNdy_Ncoll_05 = new TCanvas("c_dNdy_Ncoll_05", "AA projectile dNdy", 800, 600);

h_AA_05_dNdy_Ncoll1->GetXaxis()->SetRangeUser(-10,10);
h_AA_05_dNdy_Ncoll1->GetXaxis()->SetTitle("y");

h_AA_05_dNdy_Ncoll1->Draw("HIST same");
h_AA_05_dNdy_Ncoll2->Draw("HIST same");
h_AA_05_dNdy_Ncoll3->Draw("HIST same");
h_AA_05_dNdy_NcollL->Draw("HIST same");


auto legend_dNdy_Ncoll_05 = new TLegend(0.40,0.7,0.65,0.85);
legend_dNdy_Ncoll_05->AddEntry(h_AA_05_dNdy_Ncoll1, "Ncoll = 1", "l");
legend_dNdy_Ncoll_05->AddEntry(h_AA_05_dNdy_Ncoll2, "Ncoll = 2", "l");
legend_dNdy_Ncoll_05->AddEntry(h_AA_05_dNdy_Ncoll3, "Ncoll = 3", "l");
legend_dNdy_Ncoll_05->AddEntry(h_AA_05_dNdy_NcollL, "Ncoll = 10", "l");
legend_dNdy_Ncoll_05->Draw();
c_dNdy_Ncoll_05->Draw();

## Separate Comparison

In [ ]:
auto c_dNdy_Ncoll_cen = new TCanvas("c_dNdy_Ncoll_cen", "AA projectile dNdy", 800, 800);

h_AA_MB_dNdy_Ncoll1->GetXaxis()->SetRangeUser(-10,10);

h_AA_MB_dNdy_Ncoll1->Draw("HIST same");
h_AA_05_dNdy_Ncoll1->Draw("HIST same");


auto legend_dNdy_Ncoll_cen = new TLegend(0.15,0.7,0.4,0.85);
legend_dNdy_Ncoll_cen->AddEntry(h_AA_MB_dNdy_Ncoll1, "Ncoll = 1, MB", "l");
legend_dNdy_Ncoll_cen->AddEntry(h_AA_05_dNdy_Ncoll1, "Ncoll = 1, 0-5%", "l");

legend_dNdy_Ncoll_cen->Draw();
c_dNdy_Ncoll_cen->Draw();

In [ ]:
auto c_dNdy_Ncoll_cen = new TCanvas("c_dNdy_Ncoll_cen", "pA projectile dNdy", 800, 800);

h_AA_05_dNdy_Ncoll2->GetXaxis()->SetRangeUser(-10,10);

h_AA_05_dNdy_Ncoll2->Draw("HIST same");
h_AA_MB_dNdy_Ncoll2->Draw("HIST same");


auto legend_dNdy_Ncoll_cen = new TLegend(0.15,0.7,0.4,0.85);
legend_dNdy_Ncoll_cen->AddEntry(h_AA_MB_dNdy_Ncoll2, "Ncoll = 2, MB", "l");
legend_dNdy_Ncoll_cen->AddEntry(h_AA_05_dNdy_Ncoll2, "Ncoll = 2, 0-5%", "l");

legend_dNdy_Ncoll_cen->Draw();
c_dNdy_Ncoll_cen->Draw();

In [ ]:
auto c_dNdy_Ncoll_cen = new TCanvas("c_dNdy_Ncoll_cen", "pA projectile dNdy", 800, 800);

h_AA_MB_dNdy_Ncoll3->GetXaxis()->SetRangeUser(-10,10);

h_AA_MB_dNdy_Ncoll3->Draw("HIST same");
h_AA_05_dNdy_Ncoll3->Draw("HIST same");


auto legend_dNdy_Ncoll_cen = new TLegend(0.15,0.7,0.4,0.85);
legend_dNdy_Ncoll_cen->AddEntry(h_AA_MB_dNdy_Ncoll3, "Ncoll = 3, MB", "l");
legend_dNdy_Ncoll_cen->AddEntry(h_AA_05_dNdy_Ncoll3, "Ncoll = 3, 0-5%", "l");

legend_dNdy_Ncoll_cen->Draw();
c_dNdy_Ncoll_cen->Draw();

In [ ]:
auto c_dNdy_Ncoll_cen = new TCanvas("c_dNdy_Ncoll_cen", "pA projectile dNdy", 800, 800);

h_AA_MB_dNdy_NcollL->GetXaxis()->SetRangeUser(-10,10);

h_AA_MB_dNdy_NcollL->Draw("HIST same");
h_AA_05_dNdy_NcollL->Draw("HIST same");


auto legend_dNdy_Ncoll_cen = new TLegend(0.15,0.7,0.4,0.85);
legend_dNdy_Ncoll_cen->AddEntry(h_AA_MB_dNdy_NcollL, "Ncoll = 10 , MB", "l");
legend_dNdy_Ncoll_cen->AddEntry(h_AA_05_dNdy_NcollL, "Ncoll = 10 , 0-5%", "l");

legend_dNdy_Ncoll_cen->Draw();
c_dNdy_Ncoll_cen->Draw();

## Reason & explain

To Conclude

Each time of the collision appears to be the same,
Most of the diffences between MB and 0~2 is due to collisions larger than 3 times
This is understandable because the collision time is different


**对于任何一次都是相同的**
**是Ncoll组合的不同产生的rapidity分布的不同**

In [ ]:
auto c_dNdy_Ncoll_Compare_MB05 = new TCanvas("c_dNdy_Ncoll_Compare_MB05", "AA dNdy vs Ncoll", 1000, 500);

c_dNdy_Ncoll_Compare_MB05->Divide(2,1);
c_dNdy_Ncoll_Compare_MB05->cd(1);
h2_AA_MB_dNdy_Ncoll->Draw("colz");
c_dNdy_Ncoll_Compare_MB05->cd(2);
h2_AA_05_dNdy_Ncoll->Draw("colz");

c_dNdy_Ncoll_Compare_MB05->Draw();

auto projX_MB = h2_AA_MB_dNdy_Ncoll->ProjectionX("projX_MB");
auto projX_05 = h2_AA_05_dNdy_Ncoll->ProjectionX("projX_05");

projX_MB->SetTitle("MB");
projX_05->SetTitle("center");


auto c_Ncoll_Compare_MB05 = new TCanvas("c_Ncoll_Compare_MB052", "AA Ncoll", 1000, 500);

c_Ncoll_Compare_MB05->Divide(2,1);
c_Ncoll_Compare_MB05->cd(1);
projX_MB->Draw();
c_Ncoll_Compare_MB05->cd(2);
projX_05->Draw();

c_Ncoll_Compare_MB05->Draw();


In [ ]:
auto c_MB =  new TCanvas("c_MB","c_MB",800,800);
h2_AA_MB_dNdy_Ncoll->Draw("colz");
c_MB->Draw("projx");
